# Análisis Exploratorio de Datos (EDA) - Proyecto Criminalidad Argentina 📊🇦🇷

Este notebook está enfocado en explorar, limpiar y analizar los datos de criminalidad del **Sistema Nacional de Información Criminal (SNIC)** a nivel nacional y provincial en Argentina.

## Objetivos del EDA:
1. **Explorar las Fuentes de Datos:** Entender la estructura y variables de los datasets a nivel país y provincias.
2. **Análisis de Tendencias Temporales:** Analizar cómo ha evolucionado la criminalidad en Argentina desde el año 2000.
3. **Análisis Geográfico (Provincias):** Comparar la cantidad absoluta de delitos frente a la tasa por 100k habitantes para cada provincia.
4. **Composición de Delitos:** Identificar cuáles son las tipologías delictivas más frecuentes.
5. **Conclusiones de Negocio:** Derivar insights clave para la Gerencia Comercial (ej. hotspots de riesgo y demanda) y Técnica.

In [ ]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configurar estilo de los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Librerías importadas correctamente.")

--- 
## 1. Carga e Inspección de Datos a Nivel Nacional
Comenzaremos analizando el dataset consolidado a nivel país para observar las variables disponibles y las tendencias macro.

In [ ]:
# Rutas a los archivos de datos crudos
raw_pais_path = '../data/raw/snic-pais.csv'
raw_prov_path = '../data/raw/snic-provincias.csv'

# Verificar existencia de archivos
print("¿Existe dataset de país?:", os.path.exists(raw_pais_path))
print("¿Existe dataset de provincias?:", os.path.exists(raw_prov_path))

# Cargar datos del país
df_pais = pd.read_csv(raw_pais_path)
print(f"Dataset país cargado. Registros: {df_pais.shape[0]}, Columnas: {df_pais.shape[1]}")

In [ ]:
# Mostrar primeras filas
df_pais.head()

In [ ]:
# Información general y tipos de datos
df_pais.info()

### Observaciones del dataset nacional:
- Contiene las columnas `anio`, `codigo_delito_snic_nombre`, `cantidad_hechos` (nuestra variable principal), `cantidad_victimas`, desagregación de víctimas por sexo, y tasas calculadas por cada 100,000 habitantes (`tasa_hechos`, `tasa_victimas`).
- No hay nulos críticos en las columnas de hechos ni tasas de hechos.

--- 
## 2. Tendencia Temporal Nacional (Evolución de Delitos)
Agrupamos los datos por año para observar la cantidad total de delitos registrados a lo largo del tiempo.

In [ ]:
# Agrupar por año y sumar cantidad de hechos
delitos_por_anio = df_pais.groupby('anio')['cantidad_hechos'].sum().reset_index()

# Graficar la evolución temporal
plt.figure(figsize=(14, 6))
sns.lineplot(data=delitos_por_anio, x='anio', y='cantidad_hechos', marker='o', color='#1f77b4', linewidth=2.5)
plt.title('Evolución Histórica de Delitos Registrados en Argentina (SNIC)', fontsize=16, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Total de Delitos')
plt.xticks(delitos_por_anio['anio'].unique(), rotation=45)
plt.tight_layout()

# Guardar gráfico en carpeta images
os.makedirs('../images', exist_ok=True)
plt.savefig('../images/evolucion_temporal_nacional.png', dpi=300)
plt.show()

### Análisis del gráfico temporal:
- Hay un aumento progresivo en los delitos registrados en las últimas décadas.
- **Efecto Pandemia (2020):** Se observa una fuerte caída en la cantidad registrada en 2020 debido a las restricciones de movilidad por la cuarentena de COVID-19. 
- **Recuperación (2021+):** Post-pandemia los delitos regresan a sus niveles de tendencia anteriores. Este es un dato clave para que la Gerencia Técnica comprenda que 2020 es un outlier que nuestro modelo de series de tiempo debe manejar de forma robusta.

--- 
## 3. Tipologías Delictivas Más Frecuentes
Analizamos cuáles son las categorías de delitos que representan el mayor volumen de casos registrados en todo el país.

In [ ]:
# Agrupar por delito y sumar cantidad de hechos en todo el período
delitos_tipos = df_pais.groupby('codigo_delito_snic_nombre')['cantidad_hechos'].sum().reset_index()
delitos_tipos = delitos_tipos.sort_values(by='cantidad_hechos', ascending=False).head(10)

# Graficar el top 10
plt.figure(figsize=(12, 7))
sns.barplot(data=delitos_tipos, x='cantidad_hechos', y='codigo_delito_snic_nombre', palette='Blues_r')
plt.title('Top 10 Delitos Más Registrados en Argentina (Acumulado Histórico)', fontsize=16, fontweight='bold')
plt.xlabel('Cantidad Total de Casos')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../images/top_delitos_nacional.png', dpi=300)
plt.show()

### Insights comerciales de tipología:
- Los **Robos** y **Hurtos** dominan ampliamente las estadísticas delictivas.
- Esto valida nuestra hipótesis comercial: existe una alta demanda latente para seguros de hogar y automotor, así como sistemas de alarmas y cámaras contra robos patrimoniales.

--- 
## 4. Análisis Geográfico (Provincias)
Cargamos los datos a nivel provincial para estudiar la distribución del delito en el territorio nacional.

In [ ]:
# Cargar datos provinciales
df_prov = pd.read_csv(raw_prov_path, low_memory=False)
print(f"Dataset provincias cargado. Registros: {df_prov.shape[0]}, Columnas: {df_prov.shape[1]}")

# Limpieza básica de la variable cantidad_hechos
df_prov['cantidad_hechos'] = pd.to_numeric(df_prov['cantidad_hechos'], errors='coerce').fillna(0)
df_prov['tasa_hechos'] = pd.to_numeric(df_prov['tasa_hechos'], errors='coerce').fillna(0)

### A. Cantidad Absoluta de Delitos por Provincia
Identificamos dónde ocurre la mayor cantidad de delitos para orientar la fuerza de ventas.

In [ ]:
# Agrupar por provincia y sumar hechos
prov_absoluto = df_prov.groupby('provincia_nombre')['cantidad_hechos'].sum().reset_index()
prov_absoluto = prov_absoluto.sort_values(by='cantidad_hechos', ascending=False).head(15)

plt.figure(figsize=(12, 7))
sns.barplot(data=prov_absoluto, x='cantidad_hechos', y='provincia_nombre', palette='Oranges_r')
plt.title('Top 15 Provincias por Cantidad Absoluta de Delitos Registrados', fontsize=16, fontweight='bold')
plt.xlabel('Cantidad Total de Casos')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../images/provincias_absoluto.png', dpi=300)
plt.show()

### B. Tasa de Delitos por cada 100k Habitantes por Provincia
La cantidad absoluta suele estar sesgada por el tamaño poblacional (Buenos Aires tiene mucha más gente). Analizar la tasa por 100k habitantes nos da una visión real de la intensidad del delito en cada región (riesgo relativo), crítico para la Gerencia de Suscripción de Riesgos de seguros.

In [ ]:
# Agrupar por provincia y promediar la tasa histórica de hechos delictivos
prov_tasa = df_prov.groupby('provincia_nombre')['tasa_hechos'].mean().reset_index()
prov_tasa = prov_tasa.sort_values(by='tasa_hechos', ascending=False).head(15)

plt.figure(figsize=(12, 7))
sns.barplot(data=prov_tasa, x='tasa_hechos', y='provincia_nombre', palette='Reds_r')
plt.title('Top 15 Provincias por Tasa Promedio de Delitos (por cada 100k hab.)', fontsize=16, fontweight='bold')
plt.xlabel('Tasa de Delitos promedio por cada 100k habitantes')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../images/provincias_tasa_relativa.png', dpi=300)
plt.show()

### Revelación clave del análisis geográfico:
- **Volumen Comercial (Absoluto):** La Provincia de Buenos Aires, CABA, Córdoba y Mendoza concentran la mayor cantidad de hechos delictivos. Aquí es donde se debe desplegar la infraestructura de ventas masiva.
- **Intensidad de Riesgo (Tasa):** Provincias más pequeñas o con alta densidad urbana concentrada (ej. CABA, Neuquén, Mendoza, Santa Fe) muestran tasas de delitos por habitante más elevadas que el promedio. En estas provincias, las aseguradoras **deben incrementar las primas de cobertura** para reflejar la siniestralidad técnica real.

--- 
## 5. Conclusiones Generales del EDA
1. **Estacionalidad y Tendencias:** Se comprueba una tendencia al alza consistente, con la excepción del outlier de 2020. Un modelo clásico lineal fallaría al estimar el futuro; por ende, justificar un modelo flexible como **Prophet** es acertado.
2. **Tipología:** Los delitos patrimoniales (robos y hurtos) son mayoritarios, representando la oportunidad perfecta para comercializar pólizas de seguros contra robos y servicios de alarmas monitoreadas.
3. **Geografía:** Existe una disparidad gigante entre volumen absoluto (ventas) y tasa relativa (riesgo técnico de seguros). La Gerencia Comercial debe enfocarse en los grandes mercados (Buenos Aires/CABA) y la Gerencia Técnica debe tarificar con cuidado especial en las provincias con tasas elevadas.